In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dissertation;
CREATE SCHEMA IF NOT EXISTS dissertation.lendingclub;

BRONZE

In [0]:
# USE dissertation.lendingclub;
raw_path = "/Volumes/dissertation/lendingclub/raw/lc_2016_2017.csv"
df_raw = (spark.read
.format('csv')
# 'header','true' means the first row is the header Without this option, Spark would create columns like: _c0, _c1, _c2, _c3
.option('header','true') 
# inferSchema: Set to "true" to automatically detect column data types (e.g., Integer, String). For the raw table, we want to preserve the original dataset as much as possible Then later, in lc_clean, we convert them (hence the bronze and silve idea). Bronze / lc_raw   = raw data, minimal changes. Silver / lc_clean = cleaned and typed data. Gold / lc_ml      = ML-ready data
.option('inferSchema','false')
# This tells Spark that quotation marks are used to wrap text values.
.option('quote','\"')
# This tells Spark how to handle escaped quotation marks inside quoted text. For example, if a text field contains a quote inside it, Spark knows how to interpret it.
.option('escape','\"')
.load(raw_path)
)

# It saves the DataFrame permanently as a Databricks table. 
# .mode('overwrite') means that if the table already exists, it will be overwritten. 
# .format('delta') means that the table will be stored in the Delta Lake format. 
# .saveAsTable('lendingclub.lc_raw')
df_raw.write.mode('overwrite').format('delta').saveAsTable('dissertation.lendingclub.lc_raw')

In [0]:
display(spark.table('dissertation.lendingclub.lc_raw'))

In [0]:
spark.sql("DESCRIBE DETAIL dissertation.lendingclub.lc_raw").display()

In [0]:
spark.sql("SHOW TABLES IN dissertation.lendingclub").display()

In [0]:
display(spark.sql("""SELECT loan_amnt, emp_length, int_rate, loan_status,purpose
                  from dissertation.lendingclub.lc_raw LIMIT 10"""))

In [0]:
spark.catalog.setCurrentCatalog('dissertation')

SILVER

In [0]:

from pyspark.sql.functions import col,regexp_replace,trim

df_raw = spark.table('dissertation.lendingclub.lc_raw')
df_clean = df_raw.select(
    col('loan_amnt').cast('double').alias('loan_amnt'),
    col('funded_amnt').cast('double').alias('funded_amnt'),
    col('funded_amnt_inv').cast('double').alias('funded_amnt_inv'),

    trim(col('term')).alias('term'),
    regexp_replace(col('int_rate'),"%",'').cast('double').alias('int_rate'),
    col('installment').cast('double').alias('installment'),

    trim(col('emp_length')).alias('emp_length'),
    trim(col('home_ownership')).alias('home_ownership'),
    col('annual_inc').cast('double').alias('annual_inc'),
    trim(col('verification_status')).alias('verification_status'),
    trim(col('loan_status')).alias('loan_status'),
    trim(col('purpose')).alias('purpose'),
    trim(col('addr_state')).alias('addr_state'),
    
    col('dti').cast('double').alias('dti'),
    # credit-history count variables
    col('delinq_2yrs').cast('int').alias('delinq_2yrs'),
    col('inq_last_6mths').cast('int').alias('inq_last_6mths'),
    col('total_acc').cast('int').alias('total_acc'),
    col('open_acc').cast('int').alias('open_acc'),
    col('pub_rec').cast('int').alias('pub_rec'),

    col('revol_bal').cast('double').alias('revol_bal'),
    regexp_replace(col('revol_util'),'%','').cast('double').alias('revol_util'),
    col('collections_12_mths_ex_med').cast('int').alias('collections_12_mths_ex_med'),

    col("mths_since_last_major_derog").cast("int").alias("mths_since_last_major_derog"),
    trim(col("application_type")).alias('application_type'),
    col("acc_now_delinq").cast("int").alias("acc_now_delinq"),
    col("tot_coll_amt").cast("double").alias("tot_coll_amt"),
    col("tot_cur_bal").cast("double").alias("tot_cur_bal"),
    col("total_rev_hi_lim").cast("double").alias("total_rev_hi_lim"),
    col("inq_last_12m").cast("int").alias("inq_last_12m"),
    # repayment/future variables kept only for analysis, not ML
    col("total_pymnt").cast("double").alias("total_pymnt"),
    col("recoveries").cast("double").alias("recoveries"),
    col("last_pymnt_amnt").cast("double").alias("last_pymnt_amnt")
    )
df_clean = df_clean.filter(
    (col("loan_amnt").isNotNull()) &
    (col("annual_inc").isNotNull()) &
    (col("loan_status").isNotNull())
)
df_clean.write.mode('overwrite').format('delta').saveAsTable('dissertation.lendingclub.lc_clean')

In [0]:
display(spark.sql('DESCRIBE DETAIL dissertation.lendingclub.lc_clean'))

GOLD

In [0]:
df.select('loan_status').distinct().show()

In [0]:
from pyspark.sql.functions import when
from pyspark.sql.functions import col
df = spark.table('dissertation.lendingclub.lc_clean')
df_labeled = df.filter(col('loan_status').isin('Fully Paid','Charged Off','Default','Late (31-120 days)','Late (16-30 days)'))
# In PySpark, withColumn() is a DataFrame method used to add a new column or replace an existing column with the same name
df_labeled = df_labeled.withColumn('default_flag',when(col('loan_status').isin('Charged Off','Default','Late (31-120 days)','Late (16-30 days)'),1).otherwise(0))



In [0]:
# dataframe without leakage (excluding: grade/subgrade/int_rate)
df_ml_no_leakage = df_labeled.select('loan_amnt','funded_amnt','funded_amnt_inv','term','installment','emp_length','home_ownership','annual_inc','verification_status','purpose','addr_state','dti','delinq_2yrs','inq_last_6mths','total_acc','open_acc','pub_rec','revol_bal','revol_util','collections_12_mths_ex_med','acc_now_delinq','tot_coll_amt','tot_cur_bal','total_rev_hi_lim','inq_last_12m','default_flag')

df_ml_no_leakage.write.mode('overwrite').format('delta').saveAsTable('dissertation.lendingclub.lc_ml_no_leakage')

In [0]:
df_labeled.select('loan_status').distinct().show()

In [0]:
spark.sql('SHOW TABLES IN dissertation.lendingclub').display()

In [0]:
from pyspark.sql.functions import lit, try_divide
# Adding ratios
df_eng = spark.table('dissertation.lendingclub.lc_ml_no_leakage')
# It shows how large the loan is compared with the borrower’s yearly income.
# loan_income_ratio = how large the loan is compared with income
# installment_income_ratio = monthly repayment burden
# revol_bal_to_limit = revolving credit usage compared with credit limit
df_eng = df_eng.withColumn('loan_income_ratio',try_divide(col('loan_amnt'),col('annual_inc')))
# It shows how large the loan is compared with the borrower’s monthly
df_eng = df_eng.withColumn('installment_income_ratio',try_divide(col('installment'),col('annual_inc'))/lit(12))
df_eng = df_eng.withColumn('revol_bal_to_limit',try_divide(col('revol_bal'),col('total_rev_hi_lim')))
df_eng.write.mode('overwrite').format('delta').saveAsTable('dissertation.lendingclub.lc_ml_engineered')

In [0]:
display(spark.sql('SHOW TABLES IN dissertation.lendingclub'))

In [0]:
display(spark.sql('''SELECT default_flag, COUNT(*) AS total from dissertation.lendingclub.lc_ml_no_leakage GROUP BY default_flag'''))

In [0]:
display(spark.sql('DESCRIBE DETAIL dissertation.lendingclub.lc_ml_no_leakage'))
display(spark.sql('DESCRIBE DETAIL dissertation.lendingclub.lc_ml_engineered'))

In [0]:
spark.sql('USE CATALOG dissertation')
spark.sql('USE SCHEMA lendingclub')

In [0]:
display(spark.sql('SHOW TABLES'))

Training a basic model with less faetures to see how the model predicts on limited data

In [0]:
from pyspark.sql.functions import when, col

df = spark.table('dissertation.lendingclub.lc_clean')

df_baseline = df.withColumn('default_flag',
                             when(col('loan_status').isin(['Late (31-120 days)', 'Late (16-30 days)','Charged Off', 'Default']),1).otherwise(0)).select("loan_amnt",
                                    "funded_amnt",
                                    "funded_amnt_inv",
                                    "installment",
                                    "annual_inc",
                                    "dti",
                                    "emp_length",
                                    "revol_bal",
                                    "revol_util",
                                    "total_acc",
                                    "open_acc",
                                    "purpose",
                                    "home_ownership",
                                    "verification_status",
                                    "loan_status",
                                    "default_flag")
df_baseline.write.mode('overwrite').format('delta').saveAsTable('dissertation.lendingclub.lc_ml_baseline')
                             

Training lc_ml_baseline

In [0]:
import pandas as pd


Training Logistic Regression

In [0]:
import pandas as pd

df_pd = spark.table('dissertation.lendingclub.lc_ml_engineered').toPandas()

target = 'default_flag'

X = df_pd.drop(columns = [target])
y = df_pd[target]

# encoding categorical variables
X = pd.get_dummies(X, drop_first=True)
#  replacing the bad values made by divisions with NA. And then dropping Na rows
X = X.replace([float('inf'), -float('inf')], pd.NA).dropna()
# making sure the result dataframe doesn't contain dropped rows in X dataframe (after cleaning)
y = y.loc[X.index]

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix,roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# Standardize features by removing the mean and scaling to unit variance.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter= 3000,
                           class_weight='balanced',
                           solver='saga',
                           )
model.fit(X_train_scaled,y_train)
y_pred = model.predict(X_test_scaled)
# The model’s predicted probability that each loan belongs to class 1, meaning default.
y_prob = model.predict_proba(X_test_scaled)[:,1]

print('accuracy score',accuracy_score(y_test, y_pred))
print("AUC: ",roc_auc_score(y_test, y_prob))
print('confusion matrix: ',confusion_matrix(y_test, y_pred))
print('classification report',classification_report(y_test, y_pred))


MLflow for logging every model run

In [0]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name='logeistic_regression_balanced_n_leakage'):
    mlflow.log_param('model','LogisticRegression')
    mlflow.log_param('class_weight','balanced')
    mlflow.log_param('features_table','lc_ml_engineered')

    mlflow.log_metric('accuracy',accuracy_score(y_test,y_pred))
    mlflow.log_metric('AUC',roc_auc_score(y_test,y_prob))

    mlflow.sklearn.log_model(model,"model")


In [0]:
baseline_cols = spark.table("dissertation.lendingclub.lc_ml_baseline").columns
no_leakage_cols = spark.table("dissertation.lendingclub.lc_ml_no_leakage").columns

baseline_set = set(baseline_cols)
no_leakage_set = set(no_leakage_cols)

common_cols = sorted(baseline_set.intersection(no_leakage_set))
only_baseline = sorted(baseline_set - no_leakage_set)
only_no_leakage = sorted(no_leakage_set - baseline_set)

print("Common columns:")
print(common_cols)

print("\nOnly in lc_ml_baseline:")
print(only_baseline)

print("\nOnly in lc_ml_no_leakage:")
print(only_no_leakage)